In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown, Math
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import norm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Versión local de la librería: se busca src/pmule.py subiendo desde el directorio actual
_aqui = os.path.abspath('.')
while _aqui != os.path.dirname(_aqui) and not os.path.isfile(os.path.join(_aqui, 'src', 'pmule.py')):
    _aqui = os.path.dirname(_aqui)
sys.path.append(os.path.join(_aqui, 'src'))
from pmule import (PredecessorTable, ProjectGraph, EarnedValue,
                   pinta_tikz, plot_cargas, plot_curvas_valor_ganado,
                   tanto_por_uno, ordena_rutas)

GANTT_PARAMS = dict(arrows=True)   # flechas de prelación en los diagramas de Gantt (rojas las críticas)

In [2]:
%%capture
%mkdir -p ~/temp
%cd ~/temp

# Enunciado

Para el proyecto cuyo cuadro de prelaciones, duraciones planificadas (semanas) y presupuesto por actividad (k€) se adjunta, con el punto de control de la semana 8 que también se da, un alumno ha entregado esta resolución:

> *PV = 100, EV = 92, AC = 122. CV = AC − EV = 30 (positiva: ahorro). CPI = EV / PV = 0.92; SPI = EV / AC = 0.75. Como CPI < 1 el proyecto va retrasado y como SPI < 1 va con sobrecoste. EAC = BAC × CPI = 220 k€.*

In [3]:
#| tbl-cap: "Planificación del proyecto"
#| label: tbl-datosEnunciado
datos_enunciado = pd.DataFrame([
#  activity, predecessor, duration, BAC
    ('A'     , '---'      , 4       , 30 ),
    ('B'     , '---'      , 1       , 20 ),
    ('C'     , '---'      , 4       , 10 ),
    ('D'     , 'A,B'      , 6       , 20 ),
    ('E'     , 'C'        , 6       , 40 ),
    ('F'     , 'C,D'      , 1       , 40 ),
    ('G'     , 'E,F'      , 6       , 50 ),
    ('H'     , 'E'        , 4       , 30 ),
], columns=['activity', 'predecessor', 'duration', 'BAC']).set_index('activity')
datos_enunciado

,predecessor,duration,BAC
activity,,,
A,---,4,30
B,---,1,20
C,---,4,10
D,"A,B",6,20
E,C,6,40
F,"C,D",1,40
G,"E,F",6,50
H,E,4,30


In [4]:
#| tbl-cap: "Datos del punto de control (final de la semana 8)"
#| label: tbl-seguimiento
seguimiento = pd.DataFrame([
#  activity, actual_duration, AC, performed
    ('A'     , 6              , 39, 100      ),
    ('B'     , 1              , 30, 100      ),
    ('C'     , 4              , 15, 100      ),
    ('D'     , 2              , 6 , 25       ),
    ('E'     , 4              , 32, 67       ),
    ('F'     , 0              , 0 , 0        ),
    ('G'     , 0              , 0 , 0        ),
    ('H'     , 0              , 0 , 0        ),
], columns=['activity', 'actual_duration', 'AC', 'performed']).set_index('activity')
seguimiento

,actual_duration,AC,performed
activity,,,
A,6,39,100
B,1,30,100
C,4,15,100
D,2,6,25
E,4,32,67
F,0,0,0
G,0,0,0
H,0,0,0


In [5]:
pregunta_1 = "1. Compruebe PV, EV y AC.  "
pregunta_2 = "2. Localice todos los errores de la resolución (hay al menos cuatro) y explique cada uno.  "
pregunta_3 = "3. Escriba la resolución correcta: CV, SV, CPI, SPI, EAC y VAC, con su interpretación.  "

for item in sorted(dir()):
    if item.startswith('pregunta_'):
        display(Markdown(eval(item)))

1. Compruebe PV, EV y AC.  

2. Localice todos los errores de la resolución (hay al menos cuatro) y explique cada uno.  

3. Escriba la resolución correcta: CV, SV, CPI, SPI, EAC y VAC, con su interpretación.  

# Resolución

In [6]:
datos_proyecto = datos_enunciado.copy()
prelaciones = PredecessorTable.from_dataframe_of_strings(datos_proyecto, activity='activity', predecessor='predecessor')
proyecto = prelaciones.create_project()
BAC_total = float(datos_proyecto['BAC'].sum())
semana_control = 8
datos_valor_ganado = pd.concat([datos_proyecto, seguimiento], axis=1)
gantts = EarnedValue(proyecto).calcula_gantts(datos_valor_ganado, planned_durations_label='duration',
                                             actual_durations_label='actual_duration', PV_label='BAC', AC_label='AC',
                                             percentage_complete_label='performed', tikz=False)

## Apartado 1

In [7]:
display(Markdown(pregunta_1))

1. Compruebe PV, EV y AC.  

In [8]:
PV = float(gantts['acumulados'].loc[semana_control, 'PV'])
EV = float((datos_valor_ganado['BAC'] * datos_valor_ganado['performed'] / 100).sum())
AC = float(datos_valor_ganado['AC'].sum())
Markdown(f'PV = {PV:.0f}, EV = {EV:.0f}, AC = {AC:.0f}: los tres valores del alumno son correctos.')

PV = 100, EV = 92, AC = 122: los tres valores del alumno son correctos.

## Apartado 2

In [9]:
display(Markdown(pregunta_2))

2. Localice todos los errores de la resolución (hay al menos cuatro) y explique cada uno.  

1. **CV = AC − EV**: la variación de coste es CV = EV − AC. Con el signo cambiado un sobrecoste parece un ahorro.
2. **CPI = EV / PV**: eso es el SPI. El CPI compara valor ganado con coste real: CPI = EV / AC.
3. **SPI = EV / AC**: eso es el CPI. SPI = EV / PV.
4. **Interpretación cruzada**: CPI mide coste y SPI mide plazo; al intercambiarlos se intercambia también la lectura.
5. **EAC = BAC × CPI**: la previsión es EAC = BAC / CPI (si el rendimiento es malo, CPI < 1, el coste final es mayor, no menor).

## Apartado 3

In [10]:
display(Markdown(pregunta_3))

3. Escriba la resolución correcta: CV, SV, CPI, SPI, EAC y VAC, con su interpretación.  

In [11]:
CV, SV = EV - AC, EV - PV
CPI, SPI = EV / AC, EV / PV
EAC = BAC_total / CPI
VAC = BAC_total - EAC
pd.Series({'CV = EV − AC': CV, 'SV = EV − PV': SV, 'CPI = EV / AC': CPI, 'SPI = EV / PV': SPI, 'EAC = BAC / CPI': EAC, 'VAC = BAC − EAC': VAC}).round(3).to_frame('valor').T

,CV = EV − AC,SV = EV − PV,CPI = EV / AC,SPI = EV / PV,EAC = BAC / CPI,VAC = BAC − EAC
valor,-30.2,-8.2,0.752,0.918,318.954,-78.954


In [12]:
Markdown(f'''CPI = {CPI:.2f} ({"sobrecoste" if CPI < 1 else "ahorro"}) y SPI = {SPI:.2f} ({"retraso" if SPI < 1 else "adelanto"}). Previsión de coste final {EAC:.0f} k€ frente a {BAC_total:.0f} presupuestados (VAC = {VAC:.0f}).''')

CPI = 0.75 (sobrecoste) y SPI = 0.92 (retraso). Previsión de coste final 319 k€ frente a 240 presupuestados (VAC = -79).